# Notebook 08 — WWF RGB orthomosaics | Musanga V3_1

**Primary objective:** evaluate **native RGB quality and resolution**, and select orthomosaics for **new exhaustive Musanga annotations** across a broad forest regeneration/disturbance gradient. This is **not** a cross-validation experiment with the old UAV sites.

**Workflow:** A Setup → B GCS inventory → C survey map → D RGB native-resolution audit → E small-window visual QC → F diverse annotation site screening.

**Confirmed pilot:** `UTM33S_Plot135_RGB.tif` is EPSG:32733, **10 cm GSD**, 3 bands, 13 × 3 km and **has no overviews**. A 650-pixel preview of the *entire raster* may force huge reads. **Inspect bounded native-pixel crops instead.**

All GCS operations are read-only. No large mosaic downloads, new annotation labels or modifications to existing V3_1 model data.

## A — Environment, authentication and workspace
Install dependencies and connect Colab to Google Drive / GCS. Earth Engine is optional for the RGB-first objective.

In [ ]:
# 00 — Install dependencies
%pip -q install google-cloud-storage geopandas shapely pyogrio rasterio folium earthengine-api geemap fsspec gcsfs

In [ ]:
# 01 — Authenticate Google Cloud, mount Drive and initialize Earth Engine
from google.colab import auth, drive
from pathlib import Path
import os
auth.authenticate_user()
drive.mount('/content/drive', force_remount=False)
EE_PROJECT = os.environ.get('EE_PROJECT', '').strip()  # specify your registered GEE Cloud project if necessary
import ee
try:
    if EE_PROJECT: ee.Initialize(project=EE_PROJECT)
    else: ee.Initialize()
    print('Earth Engine ready')
except Exception as e:
    print('GEE not initialized (not required for LiDAR exploration):', repr(e))
    print('Set EE_PROJECT and rerun this cell to enable satellite comparisons.')
MUSANGA_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/Musanga')
WORK_DIR = MUSANGA_ROOT/'V3_1'/'NB08_WWF_Lidar'
WORK_DIR.mkdir(parents=True,exist_ok=True)
print('Outputs:',WORK_DIR)

## B — WWF dataset inventory
Inspect bucket object sizes, filenames and matched per-plot RGB, LiDAR and CHM coverage. Metadata only.

In [ ]:
# 02 — Bucket inventory (metadata only)
from google.cloud import storage
import pandas as pd, re
from pathlib import PurePosixPath
BUCKET='carbon-maps'
client=storage.Client()
rows=[]
for b in client.list_blobs(BUCKET):
    if b.name.endswith('/'): continue
    m=re.search(r'(UTM\d{1,2}[NS])[_/-]?(?:Plot|plot)[_-]?(\d+)',b.name,re.I)
    plot_id=f'{m.group(1).upper()}_Plot{int(m.group(2))}' if m else None
    rows.append(dict(path=b.name,extension=PurePosixPath(b.name).suffix.lower(),size_GB=(b.size or 0)/1e9,plot_id=plot_id,updated=str(b.updated)))
df=pd.DataFrame(rows)
df.to_csv(WORK_DIR/'carbon_maps_inventory.csv',index=False)
print('Files:',len(df),'GB:',round(df.size_GB.sum(),1))
display(df.groupby('extension').agg(files=('path','size'),GB=('size_GB','sum')).sort_values('GB',ascending=False).head(20))

In [ ]:
# 03 — Match WWF LiDAR products to plot IDs
patterns={
 'RGB':r'^001_Lidar_Rasters/RGB_mosaic/.*_RGB\.tif$',
 'CHM':r'^001_Lidar_Rasters/Lidar_2meter/CHM/.*\.tif$',
 'DTM':r'^001_Lidar_Rasters/Lidar_2meter/DTM/.*\.tif$',
 'DSM':r'^001_Lidar_Rasters/Lidar_2meter/DSM/.*\.tif$',
 'LAS':r'^00-raw-data/Plots/.*\.las$',
 'ECW':r'^00-raw-data/Plots/.*\.ecw$'}
products=pd.DataFrame({'plot_id':sorted(df.plot_id.dropna().unique())})
for kind,pattern in patterns.items():
    counts=df.loc[df.path.str.contains(pattern,case=False,regex=True,na=False)].groupby('plot_id').size()
    products[kind+'_n']=products.plot_id.map(counts).fillna(0).astype(int)
display(products.head(20))
print('Plots with RGB:',(products.RGB_n>0).sum())
products.to_csv(WORK_DIR/'wwf_plot_product_inventory.csv',index=False)

## C — WWF survey geography
Load the 216 georeferenced plot footprints and display a reliable offline-first survey map. Esri satellite is optional; OSM 403 errors are avoided.

In [ ]:
# 04 — Download only small plot GIS sidecars; do not read broken Ferry_Lines geometry
import tempfile
import geopandas as gpd
from pathlib import Path
prefix='003_GIS_Layers/Lidar_Plots/'
local=Path(tempfile.mkdtemp(prefix='wwf_gis_'))
for b in client.list_blobs(BUCKET,prefix=prefix):
    if b.name.lower().endswith(('.shp','.shx','.dbf','.prj','.cpg','.qpj')):
        b.download_to_filename(str(local/Path(b.name).name))
shps=list(local.glob('*.shp'))
assert shps, 'No plot shapefile found'
plots=gpd.read_file(shps[0]).to_crs('EPSG:4326')
plots=plots[plots.geometry.notna() & ~plots.geometry.is_empty].copy()
plots['geometry']=plots.geometry.make_valid()
plots['plot_id']=plots.Name.astype(str).str.strip()
plots['utm_zone']=plots.plot_id.str.extract(r'^(UTM\d{1,2}[NS])',expand=False)
plots=plots.merge(products,on='plot_id',how='left',validate='one_to_one')
print('Features:',len(plots),'Unique plots:',plots.plot_id.nunique())
display(plots.groupby('utm_zone').agg(plots=('plot_id','size'),rgb=('RGB_n',lambda x:(x>0).sum())))
plots.to_file(WORK_DIR/'wwf_lidar_plots.gpkg',driver='GPKG')

In [ ]:
# 05 — Robust interactive map (NO OpenStreetMap requests)
# The previous OSM basemap returned HTTP 403 within Colab.
# Esri satellite is optional; the default offline map always works.
import folium
from folium.plugins import Fullscreen, MiniMap
from IPython.display import display
from html import escape
import pandas as pd

# tiles=None ensures ZERO requests to restricted OSM tile servers.
m = folium.Map(location=[-2.5, 23.5], zoom_start=6, tiles=None,
               control_scale=True, prefer_canvas=True)

# Reliable fallback: blank geographic background (no external tile dependency).
folium.TileLayer(
    tiles='data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mP8/x8AAusB9Y9UxQAAAABJRU5ErkJggg==',
    attr='Local blank geographic background', name='Offline (always works)',
    overlay=False, show=True, control=True, max_zoom=19
).add_to(m)

# Optional imagery; switch it on only if your Colab browser can access Esri.
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Tiles © Esri', name='Esri World Imagery (optional)',
    overlay=False, show=False, control=True, max_zoom=19
).add_to(m)

colors={'UTM33S':'#9467bd','UTM34S':'#e3771c','UTM34N':'#2ca02c',
        'UTM35S':'#d62728','UTM35N':'#1f77b4','UTM36N':'#8c564b'}

# Plot geometry gives survey context even if ALL basemap servers fail.
foot=folium.FeatureGroup(name='Survey footprints',show=True)
folium.GeoJson(
    plots[['plot_id','geometry']].to_json(),
    style_function=lambda f: {'color':'#555555','weight':1.5,'fillOpacity':0.08},
    tooltip=folium.GeoJsonTooltip(fields=['plot_id'])
).add_to(foot)
foot.add_to(m)

for zone,group in plots.groupby('utm_zone'):
    layer=folium.FeatureGroup(name=f'{zone}: {len(group)} plots',show=True)
    for _,r in group.iterrows():
        pt=r.geometry.representative_point()
        counts={k:int(r[k+'_n']) if pd.notna(r[k+'_n']) else 0
                for k in ('RGB','LAS','CHM','DTM','DSM')}
        label=escape(str(r.plot_id))
        html='<b>'+label+'</b><br>'+'<br>'.join(f'{k}: {v}' for k,v in counts.items())
        col=colors.get(zone,'gray')
        folium.CircleMarker(
            [pt.y,pt.x],radius=5,color=col,weight=2,fill=True,
            fill_color=col if counts['RGB'] else '#ffffff',fill_opacity=.9,
            tooltip=label,popup=folium.Popup(html,max_width=300)
        ).add_to(layer)
    layer.add_to(m)

# Geographic graticule: geographic reference even with the offline background.
graticule=folium.FeatureGroup(name='Latitude / longitude grid',show=True)
for latitude in range(-12,7,2):
    folium.PolyLine([[latitude,12],[latitude,32]],color='#a0a0a0',
                    weight=.6,opacity=.4,tooltip=f'{latitude}° latitude').add_to(graticule)
for longitude in range(12,33,2):
    folium.PolyLine([[-13,longitude],[7,longitude]],color='#a0a0a0',
                    weight=.6,opacity=.4,tooltip=f'{longitude}° longitude').add_to(graticule)
graticule.add_to(m)

w,s,e,n=plots.total_bounds
m.fit_bounds([[s,w],[n,e]])
Fullscreen().add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

output=WORK_DIR/'wwf_lidar_survey_map.html'
m.save(str(output))
print('Offline-capable map:',output)
print('To see satellite imagery: select Esri World Imagery in the layer control.')
display(m)

### C.1 — Optional Musanga UAV ROIs (context only)
Their spatial overlap is **not** required. Run this cell only if you want to see the original UAV sites on the map.

In [ ]:
# 05A — Overlay the real Musanga V3_1 UAV ROI polygons on the WWF map
# Source of truth: V3_1 audited manifest and its ROI.shp (no guessed coordinates).
from pathlib import Path
from shapely.geometry import box
import geopandas as gpd
import rasterio
import folium
import pandas as pd
from IPython.display import display
import json
from html import escape

drive_root=Path('/content/drive/MyDrive')
final_root=next((q for q in [
    drive_root/'Colab Notebooks/Musanga/Final',
    drive_root/'Musanga/Final'] if q.is_dir()), None)
assert final_root, 'Musanga/Final not found. Mount Drive and check V3_1 paths.'
manifest_file=final_root/'Outputs/V3_1/00_input_audit/00_input_manifest.json'
assert manifest_file.exists(), (
    'Approved V3_1 input manifest missing: '+str(manifest_file)
    +'. Run V3_1 Notebook 00 / Block 10 first.')
manifest=json.loads(manifest_file.read_text())
roi_file=Path(manifest['roi'])
assert roi_file.is_file(), f'ROI shapefile missing: {roi_file}'
uav_roi=gpd.read_file(roi_file)
assert uav_roi.crs is not None, 'ROI.shp has no CRS; refusing to guess.'
uav_roi=uav_roi[uav_roi.geometry.notna() & ~uav_roi.geometry.is_empty].copy()
uav_roi.geometry=uav_roi.geometry.make_valid()
uav_roi=uav_roi[uav_roi.geometry.notna() & ~uav_roi.geometry.is_empty].copy()
assert len(uav_roi), 'No valid UAV ROI geometries.'

# Assign ROI polygons to a site by physical overlap with real mosaic footprints.
# Intersections are measured in the ROI's native metric CRS, not lat/lon degrees.
mosaic_records=[]
for entry in manifest['mosaics']:
    image_file=Path(entry['path'])
    if not image_file.exists():
        print('Mosaic missing; cannot assign this footprint:',image_file)
        continue
    with rasterio.open(image_file) as ds:
        assert ds.crs is not None, f'No CRS: {image_file}'
        g=gpd.GeoSeries([box(*ds.bounds)],crs=ds.crs).to_crs(uav_roi.crs).iloc[0]
        mosaic_records.append(dict(site=str(entry['site']),file=str(entry['file']),geometry=g))
assert mosaic_records, 'No readable UAV mosaics for ROI attribution.'

# Aggregate overlapping mosaics by forest site, avoiding duplicate MOS geometry.
from shapely.ops import unary_union
site_footprints={
    site:unary_union([x['geometry'] for x in mosaic_records if x['site']==site])
    for site in sorted(set(x['site'] for x in mosaic_records))}
roi_rows=[]
for index,row in uav_roi.iterrows():
    g=row.geometry
    overlaps={site:float(g.intersection(foot).area)
              for site,foot in site_footprints.items()}
    best=max(overlaps,key=overlaps.get)
    if overlaps[best]<=0:
        best='Unassigned'
    roi_rows.append(best)
uav_roi['site']=roi_rows
uav_roi['roi_row']=[int(i) for i in range(len(uav_roi))]
print('UAV ROI features:',len(uav_roi))
print('Site assignment:',uav_roi.site.value_counts().to_dict())
if (uav_roi.site=='Unassigned').any():
    print('WARNING: Unassigned ROI polygons are shown separately, not silently removed.')

roi_wgs=uav_roi.to_crs('EPSG:4326')
roi_colors={'Baego':'#ef4444','MOS':'#f59e0b','Yangambi':'#10b981',
            'Yoko':'#ec4899','Unassigned':'#111827'}
for site, group in roi_wgs.groupby('site'):
    layer=folium.FeatureGroup(name=f'UAV ROI — {site} ({len(group)})',show=True)
    color=roi_colors.get(site,'#111827')
    folium.GeoJson(
        group[['site','roi_row','geometry']].to_json(),
        style_function=lambda feat,c=color:{
            'color':c,'weight':3.5,'fillColor':c,'fillOpacity':.16},
        highlight_function=lambda feat:{'weight':5,'fillOpacity':.3},
        tooltip=folium.GeoJsonTooltip(fields=['site','roi_row'],
            aliases=['Musanga site','ROI row'])
    ).add_to(layer)
    # Distinct labels at true ROI interiors
    for _,rr in group.iterrows():
        pt=rr.geometry.representative_point()
        folium.Marker([pt.y,pt.x],
            icon=folium.DivIcon(html=(
                '<div style="font: bold 11px sans-serif;white-space:nowrap;'
                'text-shadow:1px 1px 2px white,-1px -1px 2px white;color:'
                +color+';">'+escape(site)+'</div>'))
        ).add_to(layer)
    layer.add_to(m)

# Drop old layer controls before adding the revised one.
for key, child in list(m._children.items()):
    if isinstance(child,folium.map.LayerControl):
        del m._children[key]
folium.LayerControl(collapsed=False).add_to(m)

# Plot boundaries remain visible against the complete WWF footprint.
allbounds=pd.concat([
    pd.Series([plots.total_bounds[0],roi_wgs.total_bounds[0]]),
    pd.Series([plots.total_bounds[1],roi_wgs.total_bounds[1]]),
    pd.Series([plots.total_bounds[2],roi_wgs.total_bounds[2]]),
    pd.Series([plots.total_bounds[3],roi_wgs.total_bounds[3]])
],axis=1)
m.fit_bounds([[min(plots.total_bounds[1],roi_wgs.total_bounds[1]),
               min(plots.total_bounds[0],roi_wgs.total_bounds[0])],
              [max(plots.total_bounds[3],roi_wgs.total_bounds[3]),
               max(plots.total_bounds[2],roi_wgs.total_bounds[2])]])

uav_roi.to_file(WORK_DIR/'musanga_v3_1_roi_wwf_overlay.gpkg',driver='GPKG')
map_path=WORK_DIR/'wwf_lidar_plus_musanga_uav_roi.html'
m.save(str(map_path))
print('Saved map:',map_path)
print('Saved georeferenced ROI:',WORK_DIR/'musanga_v3_1_roi_wwf_overlay.gpkg')
display(m)

## D — RGB orthomosaic native-resolution audit
Measure pixel size **from raster transforms**, not nominal filenames. Report footprint dimensions, band count, CRS, overviews, block layout and color interpretation. Avoid full-resolution reads.

In [ ]:
# D1 — Inspect a single RGB mosaic's native metadata (cheap remote header reads)
import rasterio
import pandas as pd
from rasterio.crs import CRS
from pyproj import Geod
sample_path='001_Lidar_Rasters/RGB_mosaic/UTM33S_Plot135_RGB.tif'
uri='/vsigs/'+BUCKET+'/'+sample_path
with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR', CPL_VSIL_CURL_ALLOWED_EXTENSIONS='.tif'):
    with rasterio.open(uri) as src:
        print('File:',sample_path)
        print('CRS:',src.crs)
        print('Width × height:',src.width,'×',src.height)
        print('Bands:',src.count, 'Dtypes:',src.dtypes, 'Color interpretation:',[v.name for v in src.colorinterp])
        print('Pixel size (native CRS units):',src.res)
        print('GSD in cm if CRS projected in metres:',tuple(round(100*abs(v),2) for v in src.res) if src.crs and src.crs.is_projected and src.crs.linear_units=='metre' else 'requires CRS unit conversion')
        print('Extent:',src.bounds)
        print('Internal overviews:',src.overviews(1))
        print('Storage layout:',src.profile.get('tiled'), 'Block shapes:',src.block_shapes)
        print('Compression:',src.profile.get('compress'),'Nodata:',src.nodata)
        print('Pixel count:',f'{src.width*src.height:,}')
print('NOTE: Missing overviews — NEVER downsample entire full-resolution TIFF for an overview.')

### D.2 — Metadata audit across multiple RGB mosaics
Start with **6 mosaics from different UTM zones** (modifiable), not all 216. Remote metadata access can still incur GCS requests. This saves a QC table with resolution in centimetres and flags missing overviews.

In [ ]:
# D2 — Sampled metadata audit: no image pixel reads
import math, pandas as pd, rasterio
MAX_RGB_METADATA=6
# One example per UTM zone; choose more later when working
rgb_paths=(df[df.path.str.fullmatch(r'001_Lidar_Rasters/RGB_mosaic/.*_RGB\.tif',case=False,na=False)]
           .copy())
rgb_paths['utm']=rgb_paths.plot_id.str.extract(r'^(UTM\d{1,2}[NS])',expand=False)
pick=rgb_paths.groupby('utm',sort=True).head(1).head(MAX_RGB_METADATA)
metadata_rows=[]
for rec in pick.itertuples(index=False):
    row=dict(plot_id=rec.plot_id,path=rec.path,size_GB=rec.size_GB)
    try:
        with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
            with rasterio.open('/vsigs/'+BUCKET+'/'+rec.path) as s:
                row.update(crs=str(s.crs),width=s.width,height=s.height,
                           bands=s.count,gsd_x_native=s.res[0],gsd_y_native=s.res[1],
                           overviews=len(s.overviews(1)),tiled=s.profile.get('tiled'),
                           compression=s.profile.get('compress'),
                           block_x=s.block_shapes[0][1],block_y=s.block_shapes[0][0])
                if s.crs and s.crs.is_projected and s.crs.linear_units=='metre':
                    row.update(gsd_cm_x=100*abs(s.res[0]),gsd_cm_y=100*abs(s.res[1]))
                else:
                    row.update(gsd_cm_x=float('nan'),gsd_cm_y=float('nan'))
    except Exception as e:
        row['error']=str(e)[:250]
    metadata_rows.append(row)
meta=pd.DataFrame(metadata_rows)
meta.to_csv(WORK_DIR/'wwf_rgb_metadata_audit.csv',index=False)
display(meta)
print('Saved:',WORK_DIR/'wwf_rgb_metadata_audit.csv')

## E — RGB quality inspection at **native resolution**
The main annotation decision requires individual tree crown visibility. **Do not use `src.read(out_shape=(3,650,650))` on an entire 13 km mosaic without overviews.** Instead, read **small native-pixel windows** (40 × 40 m at 10 cm = 400 × 400 pixels). Inspect a few different locations within each mosaic; the default is 4 small windows on the first pilot mosaic.

Note: a large striped TIFF can cause some extra data transfer even for a small window; inspect block shapes and monitor network usage.

In [ ]:
# E1 — RGB diagnostic: valid-data windows, true colours and native pixels
# Important: zero-valued margins are common in orthomosaics, even without nodata metadata.
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.windows import Window
from pathlib import Path
from rasterio.plot import reshape_as_image

QC_PLOT_ID = 'UTM33S_Plot135'
QC_WINDOW_METRES = 40
QC_MAX_WINDOWS = 4
QC_MAX_PROBES = 30  # bounded window reads only; no full-mosaic scan
QC_VALID_THRESHOLD = 0.80  # require at least 80% nonblack pixels
# RGB/BGR ambiguity is diagnosed by displaying both on the first selected patch.
source_rows = df[df.path.str.fullmatch(
    r'001_Lidar_Rasters/RGB_mosaic/' + QC_PLOT_ID + r'_RGB\.tif',
    case=False, na=False)]
assert len(source_rows) == 1, f'Expected exactly 1 RGB mosaic, got {len(source_rows)}'
qc_path = source_rows.iloc[0].path
preview_dir = WORK_DIR / 'RGB_QC_native_windows_v2'
preview_dir.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(8108)

with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
    with rasterio.open('/vsigs/' + BUCKET + '/' + qc_path) as src:
        assert src.crs and src.crs.is_projected and src.crs.linear_units == 'metre'
        assert src.count >= 3
        W = min(src.width, max(1, round(QC_WINDOW_METRES / abs(src.res[0]))))
        H = min(src.height, max(1, round(QC_WINDOW_METRES / abs(src.res[1]))))
        print('Raster:',qc_path)
        print('Native GSD (cm):',100*abs(src.res[0]),100*abs(src.res[1]))
        print('Band colors:',[c.name for c in src.colorinterp])
        print('Bands dtype:',src.dtypes,'Nodata:',src.nodata)
        print('Block shapes:',src.block_shapes,'Overviews:',src.overviews(1))
        print('Scanning up to',QC_MAX_PROBES,'small windows for nonblack coverage...')
        candidates = [(0.5,0.5),(0.35,0.5),(0.65,0.5),
                      (0.5,0.35),(0.5,0.65),
                      (0.25,0.25),(0.75,0.25),(0.25,0.75),(0.75,0.75)]
        candidates += list(map(tuple,rng.uniform(0.1,0.9,(QC_MAX_PROBES,2))))
        keep = []
        checked = []
        for fx, fy in candidates[:QC_MAX_PROBES]:
            col = int(np.clip(round(fx*src.width-W/2),0,src.width-W))
            row = int(np.clip(round(fy*src.height-H/2),0,src.height-H))
            win = Window(col,row,W,H)
            a = src.read(indexes=[1,2,3],window=win,masked=True)
            # A pixel is usable only if all channels are valid AND not all zeros.
            data = np.ma.filled(a,0)
            valid_mask = (~np.ma.getmaskarray(a).any(axis=0)) & np.any(data != 0,axis=0)
            good = float(valid_mask.mean())
            checked.append((col,row,round(good,3)))
            if good >= QC_VALID_THRESHOLD:
                keep.append((win,data,valid_mask,good))
                if len(keep) >= QC_MAX_WINDOWS: break
        print('Window diagnostics (col,row,valid fraction):',checked)
        if not keep:
            print('No windows meet minimum coverage. Could be sparse flight footprint or nodata encoding.')
        for i,(win,data,valid,good) in enumerate(keep,1):
            # Native integer channels: retain relative R/G/B levels (no separate 2-98% per-band stretch).
            img = np.moveaxis(data.astype('float32'),0,-1)
            vals = img[valid]
            if np.issubdtype(np.dtype(src.dtypes[0]),np.integer):
                scale=float(np.iinfo(np.dtype(src.dtypes[0])).max)
                natural=np.clip(img/scale,0,1)
            else:
                # Shared stretch across all channels; preserves relative band colours.
                hi=np.percentile(vals,99) if vals.size else 1
                natural=np.clip(img/max(hi,1e-6),0,1)
            natural[~valid] = 0
            bgr=natural[:,:,::-1]
            if i==1:
                fig,axs=plt.subplots(1,2,figsize=(12,6))
                axs[0].imshow(natural,interpolation='nearest')
                axs[0].set_title('Original band order (1,2,3)')
                axs[1].imshow(bgr,interpolation='nearest')
                axs[1].set_title('Reversed order (3,2,1) — diagnostic only')
                for ax in axs:ax.axis('off')
                plt.show();plt.close(fig)
            fig,ax=plt.subplots(figsize=(7,7))
            ax.imshow(natural,interpolation='nearest')
            ax.set_title(f'{QC_PLOT_ID} | native {100*abs(src.res[0]):g} cm | valid {good:.1%} | window {i}')
            ax.axis('off')
            output=preview_dir/f'{QC_PLOT_ID}_window{i}.png'
            fig.savefig(output,dpi=160,bbox_inches='tight')
            plt.show();plt.close(fig)
            print('Saved:',output,'pixel window:',win)
print('Judge crown delineation ONLY using valid imagery and verified band order.')


### E.2 — Manual visual quality score
Record whether crowns can be delineated, not merely nominal pixel size. Evaluate blur, shadows, mosaicking seams, color balance and the visibility of small pioneer crowns. Save a simple screening table for each inspected sample.

In [ ]:
# E2 — Editable QC sheet (fill after examining true-resolution windows)
qc_form=pd.DataFrame([{
    'plot_id':'UTM33S_Plot135','gsd_cm':10.0,
    'crown_visibility_1to5':None,'sharpness_1to5':None,
    'shadow_problem_1to5':None,'seam_problem_1to5':None,
    'possible_musanga_visible':'UNCERTAIN',
    'forest_structure':'UNREVIEWED',
    'exhaustive_annotation_feasible':'UNREVIEWED',
    'notes':'Manual review required; no species inference from GSD alone'
}])
qc_form.to_csv(WORK_DIR/'wwf_rgb_visual_qc_template.csv',index=False)
display(qc_form)
print('QC template:',WORK_DIR/'wwf_rgb_visual_qc_template.csv')

### E.3 — Six-mosaic RGB atlas for Musanga annotation feasibility
Test a deliberately small sample across UTM zones. For each mosaic, inspect up to three **35 × 35 m native-resolution crops** (with at most 12 spatial probes). Export both original band order and reversed order, because the blue/yellow colours might reflect BGR interpretation. Results include a contact sheet, individual PNGs and a CSV containing exact raster-window bounds. This is a **visual screening step**, not a validated Musanga identification or forest age label. Some mosaics may have no valid crop among the few sampled positions: record that explicitly, do not label them poor-quality. The first run makes at most 72 small window reads; large striped TIFFs may still incur extra I/O.

In [ ]:
# E3 — RGB annotation feasibility atlas: bounded native-resolution crops
# Output: 6 plots × 3 examples, individual PNGs, contact sheet, QC table.
# Zero full-mosaic reads and zero full TIFF downloads.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, rasterio
from rasterio.windows import Window, bounds as window_bounds
from pathlib import Path
from PIL import Image
from IPython.display import display
import warnings

ATLAS_PLOTS = [
    'UTM33S_Plot135','UTM34N_Plot105','UTM34S_Plot60',
    'UTM35N_Plot100','UTM35S_Plot79','UTM36N_Plot104'
]
ATLAS_CROP_M = 35                # 350 px if GSD is 10 cm
ATLAS_EXAMPLES_PER_PLOT = 3
ATLAS_MAX_PROBES_PER_PLOT = 12  # cap remote reads if flight footprint has holes
ATLAS_MIN_VALID = 0.70
# NOTE: plot-wide 1-99% stretches are not used; the 8-bit original bands are retained.
atlas_dir = WORK_DIR/'RGB_annotation_feasibility_atlas'
atlas_dir.mkdir(parents=True,exist_ok=True)

def make_display(raw, mask, dtype):
    """Both original and reversed RGB channels; identical scaling in both."""
    array = np.moveaxis(raw.astype(np.float32), 0, -1)
    good = array[mask]
    if np.issubdtype(np.dtype(dtype), np.integer):
        max_value=float(np.iinfo(np.dtype(dtype)).max)
        original=np.clip(array/max_value,0,1)
    else:
        hi=float(np.nanpercentile(good,99.5)) if good.size else 1.
        original=np.clip(array/max(hi,1e-8),0,1)
    original[~mask]=0
    return original, original[:,:,::-1]

rows=[]
images=[]
for plot_id in ATLAS_PLOTS:
    selection=df[df.path.eq(f'001_Lidar_Rasters/RGB_mosaic/{plot_id}_RGB.tif')]
    if len(selection)!=1:
        print('SKIP: missing mosaic',plot_id)
        continue
    uri='/vsigs/'+BUCKET+'/'+selection.iloc[0].path
    print('\nOPEN',plot_id,flush=True)
    try:
        with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
            with rasterio.open(uri) as ds:
                assert ds.crs and ds.crs.is_projected and ds.crs.linear_units=='metre'
                assert ds.count>=3
                nx=min(ds.width,max(1,int(round(ATLAS_CROP_M/abs(ds.res[0])))))
                ny=min(ds.height,max(1,int(round(ATLAS_CROP_M/abs(ds.res[1])))))
                print('GSD (cm):',tuple(round(abs(r)*100,2) for r in ds.res),
                      '| bands:',[c.name for c in ds.colorinterp],
                      '| overviews:',ds.overviews(1),'| blocks:',ds.block_shapes[0],flush=True)
                # Deterministic staggered probes, including central transect;
                # avoids sampling only the corners of a narrow flight swath.
                locations=[(.5,.5),(.25,.5),(.75,.5),(.5,.25),(.5,.75),
                           (.35,.35),(.65,.65),(.35,.65),(.65,.35),
                           (.15,.5),(.85,.5),(.5,.85)]
                n_saved=0
                for k,(fx,fy) in enumerate(locations[:ATLAS_MAX_PROBES_PER_PLOT]):
                    left=int(np.clip(round(fx*ds.width-nx/2),0,ds.width-nx))
                    top=int(np.clip(round(fy*ds.height-ny/2),0,ds.height-ny))
                    win=Window(left,top,nx,ny)
                    ma=ds.read(indexes=[1,2,3],window=win,masked=True)
                    raw=np.ma.filled(ma,0)
                    good=(~np.ma.getmaskarray(ma).any(axis=0)) & np.any(raw!=0,axis=0)
                    fraction=float(good.mean())
                    if fraction<ATLAS_MIN_VALID:
                        continue
                    original,reversed_rgb=make_display(raw,good,ds.dtypes[0])
                    n_saved+=1
                    rgb=(original*255).round().astype('uint8')
                    bgr=(reversed_rgb*255).round().astype('uint8')
                    stem=f'{plot_id}_crop{n_saved:02d}'
                    Image.fromarray(rgb).save(atlas_dir/f'{stem}_bands123.png')
                    Image.fromarray(bgr).save(atlas_dir/f'{stem}_bands321.png')
                    # Actual window bounding box for relocation; in source CRS
                    x0,y0,x1,y1=window_bounds(win,ds.transform)
                    rows.append(dict(plot_id=plot_id,crop=n_saved,probe=k+1,
                         x_min=x0,y_min=y0,x_max=x1,y_max=y1,
                         crs=str(ds.crs),gsd_cm_x=abs(ds.res[0])*100,
                         gsd_cm_y=abs(ds.res[1])*100,
                         valid_fraction=fraction,channel_order='UNVERIFIED',
                         crown_delineation='UNREVIEWED',musanga_identifiable='UNREVIEWED',
                         forest_environment='UNREVIEWED',notes=''))
                    images.append((stem,original,reversed_rgb,fraction))
                    print(f'  crop {n_saved}: probe {k+1}, valid {fraction:.1%}',flush=True)
                    if n_saved>=ATLAS_EXAMPLES_PER_PLOT:break
                if n_saved==0:
                    print('  WARNING: no valid crop among probed positions; not proof of bad imagery')
    except Exception as exc:
        print('ERROR inspecting',plot_id,type(exc).__name__,str(exc)[:400],flush=True)

if rows:
    table=pd.DataFrame(rows)
    table.to_csv(atlas_dir/'atlas_annotation_QC.csv',index=False)
    display(table[['plot_id','crop','gsd_cm_x','valid_fraction','channel_order',
                   'crown_delineation','musanga_identifiable']])
    # Contact sheet at modest display size only; native PNGs are retained.
    fig,axes=plt.subplots(len(images),2,figsize=(9,3.8*len(images)),squeeze=False)
    for j,(stem,a,b,valid) in enumerate(images):
        for col,img in enumerate((a,b)):
            axes[j,col].imshow(img,interpolation='nearest')
            axes[j,col].set_title(f'{stem} | '+('bands 1/2/3' if col==0 else 'bands 3/2/1'),fontsize=9)
            axes[j,col].axis('off')
    fig.suptitle('WWF RGB — native crops, identical channel scaling (colour order unverified)',fontsize=13)
    fig.tight_layout()
    fig.savefig(atlas_dir/'atlas_contact_sheet.jpg',dpi=110,bbox_inches='tight')
    display(fig);plt.close(fig)
    print('Saved contact sheet, individual native crops and QC:',atlas_dir)
else:
    print('No valid crop produced. Check nodata encoding and file layout; do NOT infer that RGB is bad.')


## F — Screening mosaics for new exhaustive annotations
Choose spatially dispersed candidate RGB mosaics as a starting list. **Spatial dispersion is not a proxy for forest age.** Classification into young secondary forest, regrowth, disturbed forest, mature canopy or open land requires image review and external land-cover history.

In [ ]:
# 08 — Catalogue RGB mosaics and select a spatially dispersed screening set
import numpy as np, pandas as pd
from pathlib import PurePosixPath
rgb=df[df.path.str.fullmatch(r'001_Lidar_Rasters/RGB_mosaic/.*_RGB\.tif',case=False,na=False)].copy()
geo=plots.copy()
cent=geo.geometry.representative_point()
geo['longitude']=cent.x;geo['latitude']=cent.y
catalog=geo[['plot_id','utm_zone','longitude','latitude','geometry']].drop(columns='geometry').merge(
    rgb[['plot_id','path','size_GB']].rename(columns={'path':'rgb_gcs_path','size_GB':'rgb_GB'}),
    how='inner',on='plot_id',validate='one_to_one')
catalog['visual_forest_class']='UNREVIEWED'
catalog['musanga_presence']='UNREVIEWED'
catalog['annotation_suitability']='UNREVIEWED'
catalog['notes']=''
catalog.to_csv(WORK_DIR/'wwf_rgb_annotation_catalog.csv',index=False)
print('RGB candidates:',len(catalog))
display(catalog.groupby('utm_zone').agg(n=('plot_id','size'),GB=('rgb_GB','sum')))
# Screen a geographically dispersed subset across all UTM zones.
# Farthest-point sampling on geographic coordinates within zones; not interpreted as age.
from pyproj import Geod
geod=Geod(ellps='WGS84')
chosen=[]
for zone, group in catalog.groupby('utm_zone'):
    target=min(5,len(group))
    pool=group.reset_index(drop=True)
    selected=[0]
    while len(selected)<target:
        distances=[]
        for j,rr in pool.iterrows():
            if j in selected: distances.append(-1);continue
            mind=min(geod.inv(rr.longitude,rr.latitude,
                              pool.iloc[k].longitude,pool.iloc[k].latitude)[2]
                     for k in selected)
            distances.append(mind)
        selected.append(int(np.argmax(distances)))
    chosen.append(pool.iloc[selected])
screen=pd.concat(chosen,ignore_index=True)
screen['review_priority']='INITIAL_SCREEN'
screen.to_csv(WORK_DIR/'wwf_rgb_initial_screen.csv',index=False)
print('Initial screening set:',len(screen),'(spatially dispersed, NOT preclassified)')
display(screen[['plot_id','utm_zone','latitude','longitude','rgb_GB']])

## G — Next decisions
Compare native GSD and observed RGB quality across zones; shortlist mosaics with visually annotatable crowns and a balanced gradient of forest environments, including Musanga-free exhaustive negatives. Expand the screening set only after reviewing the pilot true-resolution crops.

## H — TMF-guided sampling of new WWF exhaustive annotation circles

**Research aim:** expand Musanga labels to secondary and regenerating forest, degraded canopy, and recently cleared landscapes, plus undisturbed reference forest. The WWF RGB imagery is the annotation source; TMF is a **coarse (30 m), contextual sampling stratifier**, not a Musanga label or a claim about individual 125 m² circles. <https://forobs.jrc.ec.europa.eu/TMF/resources/tutorial/gee>

**Critical temporal control:** JRC TMF *annual* state must match the date of each historical WWF RGB mosaic. **The acquisition years are not yet verified.** Set a verified year for every plot before executing the candidate selection. Never use TMF 2025 to label conditions in historical images. "Recently cleared" means transition from forest to deforested land within the preceding three years, as measured in the relevant TMF annual maps. "Regrowth" is the official TMF regrowth state at the acquisition year.

**Size clarification:** a 125 **m²** circle has radius 6.31 m and diameter 12.62 m, likely too small for exhaustive crown-level annotation. The V3 exhaustive circles may instead be of a different *diameter*. The configurable pilot below defaults to the **requested 125 m²** and records that choice explicitly. Set `CIRCLE_MODE='diameter_m'` and `CIRCLE_VALUE=125` only if a **125 m diameter** is intended.

**Workflow:** verify dates → sample TMF strata within actual WWF LiDAR footprints → inspect 250 m context + native RGB validity → export selected circles and small georeferenced RGB clips to Drive → hand annotation in GIS → independent label QC. No automatic Musanga annotations or giant mosaic downloads.

In [ ]:
# H1 — Temporal metadata checklist and explicit, configurable annotation area
import math, pandas as pd
from pathlib import Path
TMF_OUT=WORK_DIR/'TMF_guided_annotation'
TMF_OUT.mkdir(parents=True,exist_ok=True)
PILOT_PLOTS=['UTM33S_Plot135','UTM34N_Plot105','UTM34S_Plot60',
             'UTM35N_Plot100','UTM35S_Plot79','UTM36N_Plot104']
CIRCLE_MODE='area_m2'  # switch to 'diameter_m' only if appropriate
CIRCLE_VALUE=125.0
CIRCLE_RADIUS_M=(math.sqrt(CIRCLE_VALUE/math.pi)
                 if CIRCLE_MODE=='area_m2' else CIRCLE_VALUE/2)
print('Circle mode:',CIRCLE_MODE,'radius (m):',round(CIRCLE_RADIUS_M,2),
      'diameter (m):',round(2*CIRCLE_RADIUS_M,2),
      'area (m²):',round(math.pi*CIRCLE_RADIUS_M**2,1))
# ENTER VERIFIED YEARS, based on acquisition logs, not export/processing timestamps.
# Each year must reflect the RGB acquisition date. None -> safely stops H2.
WWF_RGB_ACQUISITION_YEAR={p:None for p in PILOT_PLOTS}
print('WWF acquisition YEAR still to verify for:',list(WWF_RGB_ACQUISITION_YEAR))
# List potentially useful bucket reports and metadata filenames (metadata requests only)
helpful=df[df.path.str.contains(r'00-raw-data/Reports/|\.xml$|\.pdf$',case=False,regex=True)]
display(helpful[['path','size_GB']].head(55))
print('Note: filename/2016 national products do NOT establish individual RGB flight year.')

In [ ]:
# H2 — Candidate circles stratified by JRC TMF state at the RGB acquisition year
# Requires verified WWF_RGB_ACQUISITION_YEAR for each PILOT_PLOT.
import ee, geopandas as gpd, numpy as np, pandas as pd
from shapely.geometry import Point, mapping
from shapely.ops import transform
from pyproj import Transformer
assert all(WWF_RGB_ACQUISITION_YEAR[p] is not None for p in PILOT_PLOTS), (
    'STOP: fill actual WWF RGB acquisition years in H1 before selecting TMF classes.')
assert all(1993<=int(WWF_RGB_ACQUISITION_YEAR[p])<=2025 for p in PILOT_PLOTS), (
    'TMF comparison requires year 1993–2025; inspect acquisition metadata.')
TMF_COLLECTION='projects/JRC/TMF/v1_2025/AnnualChanges'
tmf=ee.ImageCollection(TMF_COLLECTION).mosaic()
SAMPLES_PER_STRATUM_PER_PLOT=5
# Category 1 undisturbed; 2 degraded; 3 recently cleared; 4 regrowth.
# Recently cleared is NOT "all deforested land": requires forest 3 years ago.
records=[]
for plot_id in PILOT_PLOTS:
    yr=int(WWF_RGB_ACQUISITION_YEAR[plot_id])
    geom=plots.loc[plots.plot_id==plot_id,'geometry'].iloc[0]
    featuregeom=ee.Geometry(mapping(geom),proj='EPSG:4326',geodesic=False)
    old=tmf.select('Dec'+str(yr-3))
    now=tmf.select('Dec'+str(yr))
    recent=now.eq(3).And(old.eq(1).Or(old.eq(2)))
    status=ee.Image(0).where(now.eq(1),1).where(now.eq(2),2)
    status=status.where(recent,3).where(now.eq(4),4).rename('stratum').toByte()
    # Sample only eligible classes within the actual LiDAR transect polygon.
    fc=status.updateMask(status.gt(0)).stratifiedSample(
        numPoints=SAMPLES_PER_STRATUM_PER_PLOT,
        classBand='stratum',
        classValues=[1,2,3,4],
        classPoints=[SAMPLES_PER_STRATUM_PER_PLOT]*4,
        region=featuregeom,scale=30,geometries=True,seed=8108,
        dropNulls=True,tileScale=4
    )
    features=fc.getInfo()['features']
    for ft in features:
        lon,lat=ft['geometry']['coordinates']
        cls=int(ft['properties']['stratum'])
        records.append(dict(plot_id=plot_id,wwf_rgb_year=yr,tmf_stratum=cls,
                            stratum_label={1:'undisturbed',2:'degraded',
                              3:'recently_cleared_3y',4:'regrowth'}[cls],
                            longitude=lon,latitude=lat))
    print(plot_id,'year',yr,'candidate centres',len(features))
assert records, 'TMF sampling returned zero candidate centres.'
candidates=pd.DataFrame(records)
candidates['candidate_id']=[f'WWF_{i:04d}' for i in range(1,len(candidates)+1)]
# Give circles exact metric area in the corresponding source UTM projection.
out=[]
for r in candidates.itertuples(index=False):
    src=plots.loc[plots.plot_id==r.plot_id].iloc[0]
    projected_crs={'UTM33S':'EPSG:32733'}  # do NOT infer from zone strings; use TIFF CRS
    rgb_path=f'001_Lidar_Rasters/RGB_mosaic/{r.plot_id}_RGB.tif'
    import rasterio
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
        with rasterio.open('/vsigs/'+BUCKET+'/'+rgb_path) as ds:
            local_crs=ds.crs
    convert=Transformer.from_crs('EPSG:4326',local_crs,always_xy=True)
    x,y=convert.transform(r.longitude,r.latitude)
    localcircle=Point(x,y).buffer(CIRCLE_RADIUS_M,resolution=32)
    # Reject sampling centres where annotation circle extends beyond mapped WWF plot geometry.
    footprint=gpd.GeoSeries([src.geometry],crs='EPSG:4326').to_crs(local_crs).iloc[0]
    if not footprint.covers(localcircle):
        continue
    out.append(dict(candidate_id=r.candidate_id,plot_id=r.plot_id,
       wwf_rgb_year=r.wwf_rgb_year,tmf_stratum=r.tmf_stratum,
       stratum_label=r.stratum_label,geometry=localcircle,
       src_crs=str(local_crs)))
# Store geometry as WGS84; geometries are reprojected from each image CRS individually.
from shapely.ops import transform as shape_transform
geo_rows=[]
for row in out:
    inverse=Transformer.from_crs(row['src_crs'],'EPSG:4326',always_xy=True)
    g=shape_transform(inverse.transform,row['geometry'])
    geo_rows.append({**{k:v for k,v in row.items() if k not in ('geometry','src_crs')},'geometry':g})
if not geo_rows:
    raise ValueError('No eligible circles wholly inside mapped WWF survey footprints.')
gdf=gpd.GeoDataFrame(geo_rows,geometry='geometry',crs='EPSG:4326')
gdf['rgb_validated']=False
gdf['visually_approved']=False
gdf['musanga_annotation_status']='NOT_ANNOTATED'
gdf.to_file(TMF_OUT/'wwf_tmf_candidate_circles.gpkg',driver='GPKG')
gdf.drop(columns='geometry').to_csv(TMF_OUT/'wwf_tmf_candidate_circles.csv',index=False)
print('Unverified candidate circles:',len(gdf))
display(gdf.groupby(['plot_id','stratum_label']).size().rename('n').reset_index())
print('NOTE: these are contextual stratification candidates, not validated local stand ages.')

In [ ]:
# H3 — Export bounded, native RGB context chips for visual review
# Select at most one candidate per plot × TMF class initially, no bulk download.
# Images contain full 250 m context; annotation circle stays much smaller.
import rasterio, numpy as np, geopandas as gpd, pandas as pd
from rasterio.windows import from_bounds, Window
from rasterio.windows import transform as window_transform
from rasterio.enums import Resampling
from pathlib import Path
CHIP_CONTEXT_M=250
MAX_CHIPS=12
RUN_CHIP_EXPORT=False  # change to True after inspecting candidate table
CHIP_DIR=TMF_OUT/'rgb_native_context_chips'
CHIP_DIR.mkdir(parents=True,exist_ok=True)
if RUN_CHIP_EXPORT:
    selection=gdf.sort_values(['plot_id','tmf_stratum','candidate_id']).groupby(
        ['plot_id','tmf_stratum']).head(1).head(MAX_CHIPS)
    chip_logs=[]
    for row in selection.itertuples(index=False):
        uri='/vsigs/'+BUCKET+f'/001_Lidar_Rasters/RGB_mosaic/{row.plot_id}_RGB.tif'
        try:
            with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
                with rasterio.open(uri) as src:
                    circle=gpd.GeoSeries([row.geometry],crs='EPSG:4326').to_crs(src.crs).iloc[0]
                    x,y=circle.centroid.x,circle.centroid.y
                    half=CHIP_CONTEXT_M/2
                    win=from_bounds(x-half,y-half,x+half,y+half,src.transform)
                    win=win.round_offsets().round_lengths()
                    full=Window(0,0,src.width,src.height)
                    win=win.intersection(full)
                    a=src.read(indexes=[1,2,3],window=win,masked=True)
                    data=np.ma.filled(a,0)
                    valid=(~np.ma.getmaskarray(a).any(axis=0)) & np.any(data!=0,axis=0)
                    fraction=float(valid.mean())
                    if fraction<.70:
                        print('SKIP low-validity',row.candidate_id,round(fraction,3))
                        continue
                    profile=src.profile.copy()
                    profile.update(driver='GTiff',count=3,width=int(win.width),
                                   height=int(win.height),transform=window_transform(win,src.transform),
                                   tiled=True,compress='deflate',BIGTIFF='IF_SAFER')
                    for key in ['photometric','interleave']:
                        profile.pop(key,None)
                    dst=CHIP_DIR/f'{row.candidate_id}_{row.plot_id}_{row.stratum_label}.tif'
                    with rasterio.open(dst,'w',**profile) as sink:
                        sink.write(data)
                        sink.colorinterp=src.colorinterp[:3]
                    chip_logs.append(dict(candidate_id=row.candidate_id,
                                          plot_id=row.plot_id,context_m=CHIP_CONTEXT_M,
                                          native_gsd_cm=100*abs(src.res[0]),
                                          valid_fraction=fraction,rgb_tif=str(dst)))
                    print('Saved:',dst.name,'| valid',round(fraction,3))
        except Exception as ex:
            print('FAILED:',row.candidate_id,repr(ex))
    pd.DataFrame(chip_logs).to_csv(TMF_OUT/'wwf_context_chip_exports.csv',index=False)
    print('Use GeoPackage circles as separate QGIS overlay. Do NOT confuse TMF stratum with a field label.')
else:
    print('Export OFF. Review H2 candidates, then set RUN_CHIP_EXPORT=True.')
print('Working directory:',TMF_OUT)

### H.4 — Human exhaustive annotation and quality controls

Load `wwf_tmf_candidate_circles.gpkg` and corresponding **georeferenced** RGB chips into QGIS. First check the imagery acquisition year, validity, and apparent disturbance/regrowth stage; reject mismatched areas. Annotate all visible Musanga crowns inside each approved circle and explicitly record true absence as zero. Keep uncertainty labels for ambiguous young crowns; do **not** treat unknown pixels as negative. Record annotator, date, annotation confidence, tree/crown counts, and image quality. For a robust cross-forest validation exercise, group spatially nearby circles by parent WWF plot to avoid leakage.

Sampling reliability: TMF pixels are ~30 m, far coarser than a **125 m²** circle. TMF determines the surrounding landscape class, not the circle's actual vegetation. Candidate status is never ground truth. Crop valid coverage and human photo interpretation must confirm suitability before annotation.

**Do not launch H2 without verified WWII/WWF acquisition dates.** If records are unavailable, first reconstruct their dates from original raw flight/processing metadata; current TMF cannot be substituted.